# 07 Manual Review Analysis

Analyze the manually reviewed pilot after the review columns are filled.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / "data" / "processed"
pilot_path = PROCESSED / "manual_review" / "manual_review_pilot.csv"

df = pd.read_csv(pilot_path)
df.shape


(103, 41)

In [2]:
df.head()


,pilot_source,id,name,category_primary,region,tier_label,place_complexity,pin_ambiguity,gt_confidence,offset_haversine_m,...,car_entry_confidence,car_entry_reasoning,entry_mode,cv_gt_lat,cv_gt_lon,cv_confidence,cv_model,cv_disagreement_m,offset_euclidean_m,offset_manhattan_m
0,high_offset,08f44a13947518d603b72a395228befa,Hayes Farms Christmas Trees,retail,FL,standard_commercial,simple,low,0.9,30.443493,...,0.2,The property appears to be a pedestrian-only u...,both,NaN,NaN,NaN,NaN,NaN,30.477737,42.888123
1,high_offset,08f44a1a94104063035aa6f92d47cc1c,Navarro Photo,pharmacy,FL,standard_commercial,simple,low,0.9,30.494132,...,0.9,The identified entry point is the most visible...,both,NaN,NaN,NaN,NaN,NaN,30.528432,42.959461
2,high_offset,08f489d59ab606c803517ccee128ff0f,Chateau Burg RV Resort,rv_park,TX,open_space,complex,high,0.9,32.981592,...,0.9,NaN,shared,NaN,NaN,NaN,NaN,NaN,33.018690,33.018690
3,high_offset,08f44d84aa9458da03f8e5d2233a32ec,FASTSIGNS,sign_making,NC,standard_commercial,simple,low,0.9,32.199257,...,0.9,The identified entry point is the most visible...,both,NaN,NaN,NaN,NaN,NaN,32.235475,39.091258
4,high_offset,08f44ae853a9ec4a030413c1cdcb4f3b,Express Factory Outlet,clothing_store,FL,standard_commercial,simple,low,0.9,30.288913,...,0.3,"The property has multiple entries, but the pri...",both,NaN,NaN,NaN,NaN,NaN,30.322982,42.670353


In [3]:
manual_cols = [
    "manual_review_status",
    "manual_should_move",
    "manual_primary_pin_type",
    "manual_needs_multi_pin",
    "manual_notes",
]

df[manual_cols].isna().sum()


manual_review_status       103
manual_should_move         103
manual_primary_pin_type    103
manual_needs_multi_pin       0
manual_notes               103
dtype: int64

In [4]:
df["manual_review_status"].value_counts(dropna=False)


manual_review_status
NaN    103
Name: count, dtype: int64

In [5]:
df["manual_should_move"].value_counts(dropna=False)


manual_should_move
NaN    103
Name: count, dtype: int64

In [6]:
df["manual_needs_multi_pin"].value_counts(dropna=False)


manual_needs_multi_pin
False    94
True      9
Name: count, dtype: int64

In [7]:
df["manual_primary_pin_type"].value_counts(dropna=False)


manual_primary_pin_type
NaN    103
Name: count, dtype: int64

In [8]:
pd.crosstab(df["pilot_source"], df["manual_review_status"], dropna=False)


manual_review_status,NaN
pilot_source,
high_offset,23
low_confidence,30
multi_tenant,25
zero_offset_sample,25


In [9]:
pd.crosstab(df["tier_label"], df["manual_review_status"], dropna=False)


manual_review_status,NaN
tier_label,
multi_tenant,26
no_building,36
open_space,6
standard_commercial,35


In [10]:
pd.crosstab(df["pin_ambiguity"], df["manual_review_status"], dropna=False)


manual_review_status,NaN
pin_ambiguity,
high,44
low,53
medium,6


In [11]:
reviewed = df[df["manual_review_status"].notna() & (df["manual_review_status"] != "")].copy()

summary = {
    "reviewed_rows": len(reviewed),
    "accepted_rate_pct": round((reviewed["manual_review_status"] == "accepted").mean() * 100, 1) if len(reviewed) else 0,
    "wrong_target_rate_pct": round((reviewed["manual_review_status"] == "wrong_target").mean() * 100, 1) if len(reviewed) else 0,
    "ambiguous_rate_pct": round((reviewed["manual_review_status"] == "ambiguous").mean() * 100, 1) if len(reviewed) else 0,
    "multi_pin_needed_rate_pct": round((reviewed["manual_needs_multi_pin"].astype(str).str.lower() == "true").mean() * 100, 1) if len(reviewed) else 0,
}

summary


{'reviewed_rows': 0,
 'accepted_rate_pct': 0,
 'wrong_target_rate_pct': 0,
 'ambiguous_rate_pct': 0,
 'multi_pin_needed_rate_pct': 0}

In [12]:
mismatch = reviewed[
    reviewed["should_move"].astype(str).str.lower()
    != reviewed["manual_should_move"].astype(str).str.lower()
].copy()

mismatch[
    [
        "pilot_source",
        "name",
        "category_primary",
        "tier_label",
        "pin_ambiguity",
        "offset_haversine_m",
        "should_move",
        "manual_should_move",
        "manual_review_status",
        "manual_notes",
    ]
].head(50)


,pilot_source,name,category_primary,tier_label,pin_ambiguity,offset_haversine_m,should_move,manual_should_move,manual_review_status,manual_notes


In [13]:
summary_path = PROCESSED / "manual_review" / "manual_review_analysis_summary.txt"

lines = [
    "Manual Review Analysis Summary",
    "",
    str(summary),
    "",
    "Review status counts",
    reviewed["manual_review_status"].value_counts(dropna=False).to_string(),
    "",
    "Manual primary pin type counts",
    reviewed["manual_primary_pin_type"].value_counts(dropna=False).to_string(),
    "",
    "Should-move mismatches",
    str(len(mismatch)),
]

summary_path.write_text("\n".join(lines) + "\n")
summary_path


PosixPath('/Users/shivanibelambe/Pin-To-Place/data/processed/manual_review_analysis_summary.txt')